In [51]:
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

True

In [52]:
import os

from anthropic import Anthropic

# Testing the Anthropic client
anthropic = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [53]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [54]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1406

In [55]:
from minsearch import Index

In [56]:
index = Index(text_fields=["question", "section", "answer"], keyword_fields=["course"])
index.fit(documents)

In [57]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1406

In [58]:
def search(question: str, course="llm-zoomcamp"):
    return index.search(
        question,
        boost_dict={"question": 2},
        filter_dict={"course": course},
        num_results=5,
    )

In [59]:
question = "I just discovered the course. Can I still join?"

In [60]:
# Adding Context
context = """
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [61]:
# Building a prompt with the search results
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know.
"""

USER_PROMPT = """
Question:
{question}

Context:
{context}
"""

In [62]:
# Function to build the context
def build_context(search_results):
    context = []
    for doc in search_results:
        context.append(doc["section"])
        context.append("Q: " + doc["question"])
        context.append("A: " + doc["answer"])
        context.append("")  # Add a blank line for better readability
    return "\n".join(context)

In [63]:
def build_prompt(question: str, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT.format(question=question, context=context)
    return prompt.strip()

In [64]:
search_results = index.search(
    question,
    boost_dict={"question": 2},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5,
)

In [65]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I still join?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: The homework submission form is still open even though the deadline has passed — can I still submit?
A: Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

General Course-Related Questions

In [66]:
def llm(prompt: str) -> str:
    """
    Simple function to call the LLM and return the response with Anthropic API.
    """
    response = anthropic.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(
        block.text
        for block in response.content
        if getattr(block, "type", None) == "text"
    )

In [67]:
response = anthropic.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=100,
    messages=[{"role": "user", "content": prompt}],
)

In [68]:
print(response.model_dump_json(indent=2))

{
  "id": "msg_011CeEvRQwx69Za1qTV1Lw2D",
  "container": null,
  "content": [
    {
      "citations": null,
      "text": "# Yes, you can still join!\n\nBased on the course information, here's what you need to know:\n\n✅ **You can join anytime** - there's no restriction on when you can start the course\n\n📋 **For a certificate:**\n- You need to submit your **Capstone project** while submissions are still being accepted\n- Homework is not mandatory for a certificate (only the Capstone project matters)\n- Homework is recommended for",
      "type": "text"
    }
  ],
  "model": "claude-haiku-4-5-20251001",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "max_tokens",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "not_available",
    "input_tokens": 348,
    "outpu

In [69]:
response.content[0].text

"# Yes, you can still join!\n\nBased on the course information, here's what you need to know:\n\n✅ **You can join anytime** - there's no restriction on when you can start the course\n\n📋 **For a certificate:**\n- You need to submit your **Capstone project** while submissions are still being accepted\n- Homework is not mandatory for a certificate (only the Capstone project matters)\n- Homework is recommended for"

In [70]:
response.usage

Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=348, output_tokens=100, output_tokens_details=None, server_tool_use=None, service_tier='standard')

In [71]:
message_history = [
    {"role": "system", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt},
]

In [72]:
response = anthropic.messages.create(
    model="claude-haiku-4-5-20251001",
    system=INSTRUCTIONS,
    max_tokens=100,
    messages=[{"role": "user", "content": prompt}],
)

In [73]:
response.content[0].text

"# Answer\n\nYes, you can still join the course! \n\nHere are the key points:\n\n- **You can start learning immediately** - there's no need to wait for confirmation or be on a registered list\n- **You can submit homework** - as long as the submission form is still open, you can submit your answers even if the deadline has passed\n- **You can earn a certificate** - to receive a certificate, you need to pass the Capstone project (homework"

In [74]:
def llm(instructions, user_prompt, model="claude-haiku-4-5-20251001", max_tokens=100):
    response = anthropic.messages.create(
        model=model,
        system=instructions,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": user_prompt}],
    )
    return "".join(
        block.text
        for block in response.content
        if getattr(block, "type", None) == "text"
    )

In [75]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt)
    return answer

In [76]:
print(rag(question))

Yes, you can still join the course! 

However, if you want to receive a certificate, you'll need to submit your Capstone project while submissions are still being accepted. Keep in mind that to earn the certificate, you need to pass the Capstone project—homework is not mandatory, though it's recommended for reinforcing concepts and contributing to your leaderboard rank.

You can start learning and submitting homework right away, and you don't need any special confirmation to


In [77]:
print(rag("How much time do I have to submit my project?"))

I don't know.

The context provided doesn't specify the exact deadline or time frame for submitting your project. It mentions that there are "attempt#1" and "attempt#2" submission windows, and that you can resubmit during attempt#2 if your first submission fails, but the specific dates or time limits are not included in the information provided.

To find out your project submission deadline, I recommend checking the **course management platform** at https://courses.
